In [4]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [5]:
#Load the trained model, scaler pickle, onehot
model = load_model(
    r"C:\Users\HP\Personal\GEN_AI\ANN_Project\model.h5"
)

In [6]:
#Load the encoder and scaler
with open(r"C:\Users\HP\Personal\GEN_AI\ANN_Project\onehot_encoder_geo.pkl", 'rb') as file:
    label_encoder_geo = pickle.load(file)

with open(r"C:\Users\HP\Personal\GEN_AI\ANN_Project\label_encoder_gender.pkl", 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open(r"C:\Users\HP\Personal\GEN_AI\ANN_Project\scaler.pkl", 'rb') as file:
    scaler = pickle.load(file)

In [7]:
#Input Data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard':1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [8]:
input_dataframe = pd.DataFrame([input_data])
input_dataframe

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
#Encoding Geography column
geography_ohe = label_encoder_geo.transform([[input_data['Geography']]])

c:\Users\HP\Personal\GEN_AI\ANN_Project\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [11]:
geography_ohe = geography_ohe.toarray()
geography_ohe

array([[1., 0., 0.]])

In [14]:
label_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [13]:
## Convert these values into dataframe having columns of geogrphy encoding
geography_ohe_df = pd.DataFrame(geography_ohe, columns=label_encoder_geo.get_feature_names_out(['Geography'])) 
geography_ohe_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [16]:
label_encoder_gender.classes_

array(['Female', 'Male'], dtype=object)

In [114]:
## Now for Gender since we have only 2 class so we'll encode this and replace value in the same df 
input_dataframe['Gender'] = label_encoder_gender.transform([input_dataframe['Gender']])
input_dataframe

c:\Users\HP\Personal\GEN_AI\ANN_Project\venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [115]:
# Drop the existing geography column and concatenete encoded ones
input_dataframe = pd.concat([input_dataframe.drop("Geography", axis=1), geography_ohe_df], axis=1)
input_dataframe

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [116]:
input_scaler_data = scaler.transform(input_dataframe)
input_scaler_data

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [117]:
prediction = model.predict(input_scaler_data)
prediction

1/1 [==============================] - 0s 232ms/step


array([[0.04728935]], dtype=float32)

In [118]:
probablity = prediction[0][0]
if (probablity > 0.5):
    print("Likely to churn")
else:
    print("Be there with Us !!")

Be there with Us !!
